In [ ]:
import simpeg.electromagnetics.time_domain as tdem
from simpeg.utils import plot_1d_layer_model, download, mkvc
from simpeg import (
    maps,
    data,
    data_misfit,
    regularization,
    optimization,
    inverse_problem,
    inversion,
    directives,
)
from discretize import TensorMesh
import empymod as empy

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
mpl.rcParams.update({"font.size":14, "font.family":"Times New Roman"})

In [ ]:
times = np.array([
    104.9, 132.7, 160.5, 188.3, 229.9,
    285.5, 341, 396.6, 452.2, 507.7,
    563.3, 618.8, 688.3, 771.6, 854.9,
    938.3, 1021.6, 1118.8, 1229.9, 1341.0,
    1466.0, 1604.9, 1743.8, 1896.6, 2063.3,
    2229.9, 2410.5, 2604.9, 2813.3, 3035.5
    ]) * 1e-6

In [ ]:
area:str = "NE"
path:str = f"./data/11-024_Alberta_{area}.csv"
picker:list = ["Line", "bheight", "TranPeak", "x_wgs84", "y_wgs84", "flight"]\
    + [f"zoff30[{i}]" for i in range(30)]

In [ ]:
# Load data
dobs_total = pd.read_csv(path)[picker]

In [ ]:
dobs_total.head()

In [ ]:
index = 0
altitude = dobs_total.iloc[index,1]
currents = dobs_total.iloc[index,2]
dobs = dobs_total.iloc[index,6:]*1e-9

In [ ]:
print(f"alt: {altitude}")
print(f"currents: {currents}")
print(f"dobs: {dobs}")

## Defining the Survey

In [ ]:
# Source loop geometry
source_location = np.array([0.0, 0.0, altitude])  # (3, ) numpy.array_like
source_orientation = "z"  # "x", "y" or "z"
source_current = currents# maximum on-time current (A)
source_radius = 5.0  # source loop radius (m)
N:int = 5 # Number of turns in the transmitter loop

# Receiver geometry
receiver_location = np.array([0.0, 0.0, altitude])  # or (N, 3) numpy.ndarray
receiver_orientation = "z"  # "x", "y" or "z"

# Receiver list
receiver_list = []
receiver_list.append(
    tdem.receivers.PointMagneticFluxTimeDerivative(
        receiver_location, times, orientation=receiver_orientation
    )
)

In [ ]:
# Define the source waveform. (unit step-off, rectangular, triangular, quarter-sine, custom)
# Triangular waveform. 
start_time = -1.74e-3
peak_time = -0.84e-3
off_time = 0.0
waveform = tdem.sources.TriangularWaveform(
    start_time,
    off_time,
    peak_time
)

In [ ]:
# Sources
source_list = [
    tdem.sources.CircularLoop(
        receiver_list=receiver_list,
        location=source_location,
        orientation=source_orientation,
        waveform=waveform,
        current=source_current,
        radius=source_radius,
        n_turns=N,
    )
]

# Survey
survey = tdem.Survey(source_list)

In [ ]:
# 5nT noise level
# tmp = 5.0*1e-9 / np.max(np.abs(dobs))
tmp = 0.05
print(tmp)
uncertainties = tmp * np.abs(dobs) * np.ones(np.shape(dobs))


In [ ]:
data_object = data.Data(survey, dobs=dobs, standard_deviation=uncertainties)

In [ ]:
# estimated host conductivity (S/m)
estimated_conductivity = 1e-2
# minimum diffusion distance
d_min = 1250 * np.sqrt(times.min() / estimated_conductivity)
print("MINIMUM DIFFUSION DISTANCE: {} m".format(d_min))
# maximum diffusion distance
d_max = 1250 * np.sqrt(times.max() / estimated_conductivity)
print("MAXIMUM DIFFUSION DISTANCE: {} m".format(d_max))

In [ ]:
depth_min = 5  # top layer thickness
depth_max = 100.0  # depth to lowest layer
geometric_factor = 1.15  # rate of thickness increase

In [ ]:
# Increase subsequent layer thicknesses by the geometric factors until
# it reaches the maximum layer depth.
layer_thicknesses = [depth_min]
while np.sum(layer_thicknesses) < depth_max:
    layer_thicknesses.append(geometric_factor * layer_thicknesses[-1])

n_layers = len(layer_thicknesses) + 1  # Number of layers

In [ ]:
# m = log(conductivity)  
# log_conductivity_map = maps.ExpMap(nP=n_layers) # nP is the number of layers

l_bound = 1e-3
u_bound = 1e-1

log_conductivity_map = maps.LogisticSigmoidMap(nP=n_layers, lower_bound=l_bound, upper_bound=u_bound) # nP is the number of layers

In [ ]:
# Starting model is log-conductivity values (S/m)
starting_cond = 5e-2
# starting_conductivity_model = np.log(7e-2 * np.ones(n_layers))
starting_conductivity_model = np.log((starting_cond -l_bound)/(u_bound - starting_cond) * np.ones(n_layers))
# Reference model is also log-resistivity values (S/m)
reference_conductivity_model = starting_conductivity_model.copy()
# reference_conductivity_model = np.log(1e-2 * np.ones(n_layers))

In [ ]:
simulation_L2 = tdem.Simulation1DLayered(
    survey=survey, thicknesses=layer_thicknesses, sigmaMap=log_conductivity_map
)

In [ ]:
dpred_data = simulation_L2.dpred(starting_conductivity_model)

In [ ]:
fig = plt.figure(figsize=(5, 5))
ax = fig.add_axes([0.15, 0.15, 0.8, 0.75])
ax.loglog(times, np.abs(dobs), "k-o", lw=3)
ax.loglog(times, np.abs(dpred_data), "b-o", lw=3)
ax.grid(which="both")
ax.set_xlabel("Times (s)")
ax.set_ylabel("|B| (T/s)")
ax.set_title("Observed Data")
plt.show()

In [ ]:
dmis_L2 = data_misfit.L2DataMisfit(simulation=simulation_L2, data=data_object)

In [ ]:
# Define 1D cell widths
h = np.r_[layer_thicknesses, layer_thicknesses[-1]]
h = np.flipud(h)

# Create regularization mesh
regularization_mesh = TensorMesh([h], "N")
print(regularization_mesh)

In [ ]:
reg_L2 = regularization.WeightedLeastSquares(
    regularization_mesh,
    alpha_s=1,
    # alpha_x=0.0, # weighting for the first derivative term
    length_scale_x=10.0,
    reference_model=reference_conductivity_model,# Log resistivity
    reference_model_in_smooth=False,
)

In [ ]:
opt_L2 = optimization.InexactGaussNewton(
    maxIter=100, maxIterLS=20, cg_maxiter=20, cg_rtol=1e-3
)

$$
\phi(m)=\phi_{d}(m)+\beta\cdot\phi_m(m)
$$

**Trade-off Parameter:** $\beta_0 = \text{beta0\_ratio} \cdot \frac{\lambda_{max}(J^T W_d^T W_d J)}{\lambda_{max}(W_m^T W_m)}$

In [ ]:
inv_prob_L2 = inverse_problem.BaseInvProblem(dmis_L2, reg_L2, opt_L2)

In [ ]:
# Inversion Directives

update_jacobi = directives.UpdatePreconditioner(update_every_iteration=True)
# Initial beta
starting_beta = directives.BetaEstimate_ByEig(beta0_ratio=5)
# Beta schedule: Decrease beta by a factor of 2 every 3 iterations
beta_schedule = directives.BetaSchedule(coolingFactor=2.0, coolingRate=3)
target_misfit = directives.TargetMisfit(chifact=1.0)

directives_list_L2 = [update_jacobi, starting_beta, beta_schedule, target_misfit]

In [ ]:
# Here we combine the inverse problem and the set of directives
inv_L2 = inversion.BaseInversion(inv_prob_L2, directives_list_L2)

# Run the inversion
recovered_model_L2 = inv_L2.run(starting_conductivity_model)

# Output Table Columns:
# #:Iteration number
# beta: Trade-off parameter (weighting of the regularization)
# phi_d: Data misfit
# phi_m: Model misfit
# f: Total objective function value (phi_d + beta * phi_m)
# |proj(x-g)-x|: Gradient projection (convergence indicator)
# LS: Line search iterations 

In [ ]:
dpred_L2 = simulation_L2.dpred(recovered_model_L2)

fig = plt.figure(figsize=(5, 5))
ax1 = fig.add_axes([0.15, 0.15, 0.8, 0.75])
ax1.loglog(times, np.abs(dobs), "k-o")
ax1.loglog(times, np.abs(dpred_L2), "b-o")
ax1.grid(which="both")
ax1.set_xlabel("times (s)")
ax1.set_ylabel("Bz (T/s)")
ax1.set_title("Predicted and Observed Data")
ax1.legend(["Observed", "L2 Inversion"], loc="upper right")
plt.show()

In [ ]:
recovered_conductivities = log_conductivity_map * recovered_model_L2

In [ ]:
def inverse_map(x):
    return (u_bound - l_bound) / (1 + np.exp(-x)) + l_bound

In [ ]:
recovered_conductivities = inverse_map(recovered_conductivities)

In [ ]:
final_misfit = dmis_L2(recovered_model_L2)
chi_squared = final_misfit / times.size
print(f"{chi_squared=:.2f}")

In [ ]:
# Plot true model and recovered model
fig = plt.figure(figsize=(6, 6))

ax1 = fig.add_axes([0.2, 0.15, 0.7, 0.7])
plot_1d_layer_model(
    layer_thicknesses, 1/recovered_conductivities, ax=ax1, color="b"
)
ax1.grid()
ax1.set_xlabel(r"Resistivity ($\Omega m$)")
ax1.legend(["True Model", "L2-Model"])
plt.show()

In [ ]:
1/recovered_conductivities